# Creating simulated datasets

This is step 1 of the ISO end-to-end pipeline:

1. **this notebook** synthesises the datasets,
2. [`create_cross_covariance.ipynb`](create_cross_covariance.ipynb) builds the cross-covariance
   between them,
3. [`analyse_datasets.ipynb`](analyse_datasets.ipynb) runs the joint fit on exactly these products.

Every fiducial value, setting and file path comes from [`defaults/`](defaults/README.md), read
through `iso_config.py`. The products go to the git-ignored `sims/` directory, under the names set
in `defaults/paths.yaml`, which the downstream notebooks read. We build, from simplest to most
involved:

1. **From scratch**: a galaxy x CMB-lensing cross-correlation, computed with
   [CCL](https://github.com/LSSTDESC/CCL), with the cosmology and the noise exposed as knobs.
2. **A smooth CMB-lensing twin**: the shipped reconstruction with its bandpowers replaced by the
   theory at the ISO fiducial (chi-square = 0 when fit at the same cosmology), via
   `soliket.sacc_tools.smooth_twin_sacc`.
3. **A smooth MFLike CMB+foregrounds dataset**: the per-frequency theory binned through MFLike's own
   windows, via `soliket.sacc_tools.smooth_mflike_sacc`.

Datasets 2 and 3 are the joint-analysis inputs. Everything is built at the **ISO fiducial** of
`defaults/cosmo.yaml`, including the SO normal-hierarchy neutrinos.

In [ ]:
import numpy as np
import sacc

import iso_config

# File locations come from defaults/paths.yaml (shared with the other ISO notebooks).
SIMS = iso_config.path("sims_dir")
SIMS.mkdir(exist_ok=True)

GALAXY_KAPPA = iso_config.path("galaxy_kappa")
LENSING_SMOOTH = iso_config.path("lensing_smooth")
MFLIKE_SMOOTH = iso_config.path("mflike_smooth")
print("writing simulated datasets to", SIMS)

## 1. A galaxy x kappa dataset from scratch

We simulate the cross-correlation of an unWISE-like galaxy sample with the SO CMB-lensing
convergence. The three spectra (`gg`, `gk`, `kk`) come from CCL; the bandpower windows and the joint
covariance come from `soliket.sacc_tools`. The sample and binning settings are in
`defaults/sims/galaxy_kappa.yaml` and the cosmology is the ISO fiducial
(`iso_config.ccl_cosmology()`). We wrap it in a function so the cosmology and the galaxy noise can
be changed.

In [ ]:
import pyccl as ccl

from soliket.sacc_tools import gaussian_covariance, top_hat_windows

# Galaxy sample and binning settings.
GK = iso_config.load_yaml("sims/galaxy_kappa.yaml")

# Fiducial galaxy redshift distribution, borrowed from the shipped reference dataset.
ref = sacc.Sacc.load_fits(str(iso_config.path("galaxy_nz_reference")))
Z, NZ = ref.tracers["gc_unwise"].z, ref.tracers["gc_unwise"].nz


def build_galaxy_kappa(out_path, *, cosmo, noise_gg=None, settings=GK):
    """Simulate an unWISE-like galaxy x SO CMB-lensing cross-correlation SACC.

    `cosmo` is a ``ccl.Cosmology``; `noise_gg` optionally overrides the galaxy
    auto-spectrum noise (default: shot noise ``1 / n_gal``). Returns the SACC.
    """
    b1 = settings["galaxy_bias"]
    mag_bias = settings["magnification_bias"]
    ngal_per_arcmin2 = settings["ngal_per_arcmin2"]
    ell_max = settings["ell_max"]
    n_bins = settings["n_bins"]

    gc = ccl.NumberCountsTracer(
        cosmo,
        has_rsd=False,
        dndz=(Z, NZ),
        bias=(Z, b1 * np.ones_like(Z)),
        mag_bias=(Z, mag_bias * np.ones_like(Z)),
    )
    ck = ccl.CMBLensingTracer(cosmo, z_source=settings["z_source_cmb"])

    ells, window = top_hat_windows(ell_max, n_bins)
    delta_ell = ell_max // n_bins
    cl_gg = ccl.angular_cl(cosmo, gc, gc, ells)
    cl_gk = ccl.angular_cl(cosmo, gc, ck, ells)
    cl_kk = ccl.angular_cl(cosmo, ck, ck, ells)

    if noise_gg is None:  # default: galaxy shot noise 1 / n_gal
        ngal_sr = ngal_per_arcmin2 / np.deg2rad(1.0 / 60.0) ** 2
        noise_gg = np.full_like(cl_gg, 1.0 / ngal_sr)
    cls = np.array([[cl_gg + noise_gg, cl_gk], [cl_gk, cl_kk]])
    cov = gaussian_covariance(cls, ells, delta_ell, settings["fsky"])

    s = sacc.Sacc()
    s.metadata["info"] = "Simulated unWISE-like galaxy x SO CMB-lensing cross-correlation"
    s.add_tracer(
        "NZ",
        "gc_unwise",
        quantity="galaxy_density",
        spin=0,
        z=Z,
        nz=NZ,
        metadata={"ngal": ngal_per_arcmin2},
    )
    s.add_tracer(
        "Map",
        "ck_so",
        quantity="cmb_convergence",
        spin=0,
        ell=np.arange(3000),
        beam=np.ones(3000),
    )
    s.add_ell_cl("cl_00", "gc_unwise", "gc_unwise", ells, cl_gg, window=window)
    s.add_ell_cl("cl_00", "gc_unwise", "ck_so", ells, cl_gk, window=window)
    s.add_ell_cl("cl_00", "ck_so", "ck_so", ells, cl_kk, window=window)
    s.add_covariance(cov)
    s.save_fits(str(out_path), overwrite=True)
    return s

In [ ]:
# Canonical dataset at the ISO fiducial cosmology.
fiducial = iso_config.ccl_cosmology()
s = build_galaxy_kappa(GALAXY_KAPPA, cosmo=fiducial)
print("wrote", GALAXY_KAPPA.name, "->", len(s.mean), "data points")

# Knob 1 - a different cosmology (lower A_s -> lower sigma8).
low_amp = iso_config.ccl_cosmology(A_s=1.8e-9)
build_galaxy_kappa(SIMS / "galaxy_kappa.lowA.fits", cosmo=low_amp)

# Knob 2 - a custom (flat, noisier) galaxy noise spectrum instead of pure shot noise.
ells, _ = top_hat_windows(GK["ell_max"], GK["n_bins"])
build_galaxy_kappa(
    SIMS / "galaxy_kappa.noisy.fits", cosmo=fiducial, noise_gg=np.full(len(ells), 5e-6)
)
print("knob variants written (different cosmology, custom noise)")

## 2. A smooth CMB-lensing twin

A *smooth* twin reuses the shipped dataset's tracers, bandpower windows and covariance but replaces
the noisy measured bandpowers with the theory at a chosen cosmology, so the likelihood gives
chi-square = 0 when fit at that cosmology. `soliket.sacc_tools.smooth_twin_sacc` does the SACC
surgery; we get the binned theory from an evaluated `lensing` likelihood, built with
`iso_config.build_info("lensing", data="shipped")` (the shipped data provides the windows and
covariance) and every parameter fixed to the fiducial. The **imprint cosmology is a knob**: by
default the ISO fiducial, but any param override (e.g. a shifted `tau`) imprints a non-fiducial
twin for parameter-recovery tests.

In [ ]:
from cobaya.model import get_model
from cobaya.tools import resolve_packages_path

from soliket.presets import resolve_aliases
from soliket.sacc_tools import smooth_twin_sacc


def smooth_lensing(out_path, **param_overrides):
    """Write a smooth (theory) CMB-lensing twin at the ISO fiducial.

    `param_overrides` pins fiducial params (e.g. ``tau=0.06``) to imprint the twin
    at a non-fiducial cosmology. Returns ``(lensing_likelihood, binned_clkk)``.
    """
    info = iso_config.build_info("lensing", data="shipped")
    info["params"] = iso_config.fix_to_fiducial(info["params"])
    for name, value in param_overrides.items():
        info["params"][name] = {"value": value}
    info["packages_path"] = resolve_packages_path()

    model = get_model(info)
    model.loglikes({})  # evaluate at the imprint cosmology
    lensing = resolve_aliases(model).lensing
    clkk = lensing._get_theory()  # binned C_ell^kappakappa

    src = sacc.Sacc.load_fits(lensing.datapath)  # reuse shipped tracers/windows/cov
    smooth_twin_sacc(src, "cl_00", "ck", "ck", clkk, out_path=out_path)
    return lensing, clkk


lensing, clkk = smooth_lensing(LENSING_SMOOTH)
print("wrote", LENSING_SMOOTH.name, "->", len(clkk), "bins")

In [ ]:
# Knob - imprint a second twin at a shifted tau (a parameter-recovery target).
smooth_lensing(SIMS / "lensing_smooth.tau0p06.sacc.fits", tau=0.06)
print("wrote a tau=0.06 twin alongside")

## 3. A smooth MFLike CMB + foregrounds dataset

The same smooth-twin idea for the primary CMB, where the data vector spans many frequency
cross-spectra. The per-frequency plumbing — combining CMB + foregrounds + systematics through
MFLike's `get_modified_theory`, binning with MFLike's own bandpower windows, and writing one `NuMap`
tracer per `(frequency, spin)` channel — lives in `soliket.sacc_tools.smooth_mflike_sacc`, which
takes the concrete handles (the evaluated likelihood + theory outputs, not a `Session`).

This build runs CAMB at MFLike accuracy and takes a few minutes; the covariance and bandpower-window
(Bbl) matrices are reused from the shipped `cov_Bbl_file`, so only the data vector is regenerated.

In [ ]:
from soliket.sacc_tools import smooth_mflike_sacc

RUN_MFLIKE = True  # set False to skip the few-minute CAMB build

if RUN_MFLIKE:
    # The shipped MFLike data provides the windows and covariance of the twin.
    info = iso_config.build_info("mflike", data="shipped")
    info["params"] = iso_config.fix_to_fiducial(info["params"])
    info["packages_path"] = resolve_packages_path()
    model = get_model(info)
    roles = resolve_aliases(model)

    params = iso_config.fiducial_values(info["params"])  # numeric fiducial values
    model.loglikes(params)  # evaluate at the ISO fiducial

    dls = model.provider.get_Cl(ell_factor=True)
    fg_totals = roles.foreground.get_fg_totals()
    smooth_mflike_sacc(roles.mflike, dls, fg_totals, params, out_path=MFLIKE_SMOOTH)
    print("wrote", MFLIKE_SMOOTH.name)
else:
    print("RUN_MFLIKE is False - skipping the smooth MFLike build.")

## Recap

`sims/` now holds the pipeline inputs:

| Artifact | Built from | Consumed by |
| --- | --- | --- |
| `mflike_smooth.fits` | `smooth_mflike_sacc` (CMB+fg theory, ISO fiducial) | the analysis (MFLike component) |
| `lensing_smooth.sacc.fits` | `smooth_twin_sacc` (lensing theory, ISO fiducial) | the analysis (lensing component) |
| `galaxy_kappa.sim.fits` (+ knob variants) | CCL from scratch | standalone galaxy x kappa example |

The smooth lensing twin carries only the data + covariance; `defaults/probes/lensing/sims.yaml`
points the likelihood's `correction_filename` / `fiducial_filename` at the shipped
(multi-hundred-MB) auxiliary files in place, so we never copy those into `sims/`. Both smooth twins
are built at the **same ISO fiducial** the analysis fits at, so the joint chi-square is 0 by
construction.

Next: [`create_cross_covariance.ipynb`](create_cross_covariance.ipynb) builds the cross-covariance
between the MFLike and lensing data vectors; then
[`analyse_datasets.ipynb`](analyse_datasets.ipynb) runs the joint fit on these `sims/` products.